In [1]:
import pandas as pd
data= pd.read_csv("rxdx_aug_2026_indexed_data_med_mod.csv")
data.head()

,description,NDC,HCPCS
0,"""HYDROCODONE-ACETAMINOPHEN 7.5-325 MG/15 ML SO...",53217-0136-01,NaN
1,"""HYDROCODONE-ACETAMINOPHEN 7.5-325 MG/15 ML SO...",53217-0136-01,NaN
2,(DAUNORUBICIN AND CYTARABINE) LIPOSOME 100 MG/...,68727-0745-02,J9100
3,(DAUNORUBICIN AND CYTARABINE) LIPOSOME 100 MG/...,68727-0745-02,NaN
4,"(TO DELIVER) AVOBENZONE 2%, HOMOSALATE 8%, OCT...",66800-3351-07,NaN


In [2]:
import os
import shutil
import pandas as pd

from whoosh import index
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import SimpleAnalyzer

In [3]:
INDEX_DIR = r"C:\temp\whoosh_index"

In [4]:
if os.path.exists(INDEX_DIR):

    shutil.rmtree(INDEX_DIR, ignore_errors=True)

In [5]:
os.makedirs(INDEX_DIR, exist_ok=True)

In [7]:
schema = Schema(

    doc_id=ID(stored=True, unique=True),

    ndc=ID(stored=True),

    hcpcs=ID(stored=True),

    description=TEXT(stored=True),

    content=TEXT(
        stored=True,
        analyzer=SimpleAnalyzer()
    )
)

In [8]:
ix = index.create_in(INDEX_DIR, schema)

In [9]:
df = data.fillna("")

In [10]:
writer = ix.writer(limitmb=512)

In [12]:
for idx, row in enumerate(df.itertuples(index=False)):

    ndc = str(row.NDC)

    hcpcs = str(row.HCPCS)

    description = str(row.description)

    # ONLY codes searchable
    searchable_text = f"{ndc} {hcpcs}"

    writer.add_document(

        doc_id=str(idx),

        ndc=ndc,

        hcpcs=hcpcs,

        description=description,

        content=searchable_text
    )

writer.commit()

In [13]:

from whoosh import index

ix = index.open_dir(r"C:\temp\whoosh_index")

In [14]:
searcher = ix.searcher()

In [15]:
from whoosh.qparser import MultifieldParser, OrGroup

In [16]:
parser = MultifieldParser(
    ["description","doc_id", "ndc", "hcpcs", "content"],
    schema=ix.schema,
    group=OrGroup
)

In [20]:
query = parser.parse("Aneshthesia")

In [21]:
results = searcher.search(query, limit=5)
print(results)

<Top 0 Results for Or([Term('description', 'aneshthesia'), Term('doc_id', 'Aneshthesia'), Term('ndc', 'Aneshthesia'), Term('hcpcs', 'Aneshthesia'), Term('content', 'aneshthesia')]) runtime=0.0020504999847617>


In [22]:
for r in results:

    print("=" * 50)

    print("doc_id:", r["doc_id"])
    
    print("Description:", r["description"])

    print("NDC:", r["ndc"])

    print("HCPCS:", r["hcpcs"])

    print("Score:", r.score)